In [6]:
import sys
from dataclasses import dataclass

import neuspell
from neuspell import BertChecker  # or SclstmChecker for faster LSTM

In [7]:
import torch

# monkeypatch for pytorch 2.6
_orig_torch_load = torch.load

def _torch_load_legacy(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_legacy

In [8]:
@dataclass
class SpellCorrector:
    checker: BertChecker

    @classmethod
    def load(cls) -> "SpellCorrector":
        print("load neuspell bert checker…")
        checker = BertChecker()
        checker.from_pretrained()
        print("model loaded.\n")
        return cls(checker=checker)

    def correct(self, text: str) -> str:
        return self.checker.correct(text)

In [9]:
def repl(corrector: SpellCorrector) -> None:
    print("neuspell spell correction test")
    print("insert query (empty + enter to finish).\n")

    while True:
        try:
            query = input("query> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            return

        if not query:
            print("Finish.")
            return

        corrected = corrector.correct(query)

        print(f"original : {query}")
        print(f"corrected: {corrected}")
        print("-" * 40)

In [10]:
def main() -> None:
    corrector = SpellCorrector.load()
    repl(corrector)


if __name__ == "__main__":
    sys.exit(main())

load neuspell bert checker…
loading vocab from path:/home/vietc/projects/search-engine/src/backend/.venv/lib/python3.13/site-packages/neuspell/../data/checkpoints/subwordbert-probwordnoise/vocab.pkl
initializing model
SubwordBert(
  (bert_dropout): Dropout(p=0.2, inplace=False)
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_featu

UnpicklingError: invalid load key, '<'.